# 规范 Chapter 5: Model Evaluation and Improvement
**Reference:** *Introduction to Machine Learning with Python: A Guide for Data Scientists* by Andreas C. Müller & Sarah Guido

---

## 5.1 Introduction to Robust Model Evaluation
Up to this point, we have evaluated our machine learning models using the standard `train_test_split` method. While this approach provides a quick baseline, it can be highly deceptive. The generalization score returned by a single split is heavily dependent on how the random number generator shuffled the rows before partitioning. If a particular data configuration places easy-to-classify samples in the test set, the accuracy will be artificially inflated.

To build systems capable of robust real-world generalization, we must use data-driven statistical evaluation. This chapter focuses on **Cross-Validation**, automated hyperparameter optimization via **Grid Search**, and advanced **Evaluation Metrics** designed to measure performance when dealing with imbalanced datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import train_test_split

# Set visual rendering engine parameters
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
print("Model evaluation environment successfully initialized.")

## 5.2 Cross-Validation
Instead of splitting the dataset into training and test sets just once, **K-Fold Cross-Validation** splits the data into $K$ equally sized, non-overlapping segments called folds. The algorithm then performs a loop of $K$ distinct training and evaluation cycles.

In the first iteration, Fold 1 acts as the test set (validation set), while Folds 2 through $K$ are combined to train the model. In the next iteration, Fold 2 is held out as the test set, while the remaining folds are used for training. This process repeats until every single fold has served as the validation space exactly once. 

By averaging the resulting $K$ accuracy scores, we obtain a highly stable, low-variance estimate of how well the model generalizes to completely unseen data partitions.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression

# Load the baseline multi-class Iris dataset
iris = load_iris()
logreg = LogisticRegression(max_iter=1000)

# Execute a 5-Fold Cross Validation sequence
# cross_val_score automatically handles the data segmentation loop internally
cv_scores = cross_val_score(logreg, iris.data, iris.target, cv=5)

print("Cross-Validation Scores per fold:", cv_scores)
print(f"Mean Generalization Accuracy:     {cv_scores.mean():.3f}")

# Visualize the metric stability across different cross-validation partitions
plt.figure(figsize=(10, 5))
plt.bar(range(1, 6), cv_scores, color='skyblue', edgecolor='black', alpha=0.8)
plt.axhline(cv_scores.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean Accuracy ({cv_scores.mean():.2f})')

plt.ylim(0.8, 1.05)
plt.xlabel('Cross-Validation Fold Index', fontsize=12)
plt.ylabel('Classification Accuracy', fontsize=12)
plt.title('Logistic Regression Stability Across 5-Fold Cross-Validation Columns', fontsize=14)
plt.legend(loc='lower right')
plt.show()

## 5.3 Grid Search (Hyperparameter Tuning)
Finding the optimal settings for a model's hyperparameters (such as the regularization parameter `C` and kernel width `gamma` in a Support Vector Machine) is crucial for maximizing prediction accuracy. Testing these combinations manually by rewriting training code is highly inefficient.

To automate this workflow, we use **Grid Search**. We construct a programmatic grid representing all combinations of parameters we wish to evaluate. Scikit-learn's `GridSearchCV` evaluates every single parameter intersection using cross-validation. 

### The Train-Validation-Test Workflow
It is vital to separate your hyperparameter search from your final model validation. If you select parameters based on your test set scores, you are guilty of **Data Leakage**—the parameters will have memorized the test set configuration, leading to overly optimistic performance estimates. `GridSearchCV` avoids this by splitting the training subset further during its internal loops.

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# Construct the target Parameter Grid mapping a logarithmic space
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1, 10, 100]
}

# Isolate a secure evaluation Test Set using train_test_split before tuning
X_train, X_test, y_train, y_test = train_test_split(iris.data, iris.target, random_state=0)

# Instantiate the GridSearchCV object with an internal 5-fold cross-validation loop
grid_search = GridSearchCV(SVC(), param_grid, cv=5, return_train_score=True)

# Run the full search matrix (6 x 6 = 36 distinct parameter configurations evaluated)
grid_search.fit(X_train, y_train)

print(f"Optimal Parameter Configuration Found: {grid_search.best_params_}")
print(f"Best Internal Cross-Validation Score:   {grid_search.best_score_:.3f}")

# Extract and format structural search results into a spatial matrix for heat mapping
search_results = pd.DataFrame(grid_search.cv_results_)
scores_matrix = np.array(search_results.mean_test_score).reshape(6, 6)

# Draw the parameter search heatmap
plt.figure(figsize=(8, 6))
plt.imshow(scores_matrix, interpolation='nearest', cmap='viridis')
plt.xlabel('Hyperparameter: gamma', fontsize=12)
plt.ylabel('Hyperparameter: C', fontsize=12)
plt.colorbar(label='Cross-Validated Accuracy Mean')
plt.xticks(np.arange(6), param_grid['gamma'])
plt.yticks(np.arange(6), param_grid['C'])
plt.title('SVM Hyperparameter Optimization Matrix (Grid Search Heatmap)', fontsize=14)
plt.show()

## 5.4 Evaluation Metrics for Imbalanced Datasets
When real-world datasets are heavily **imbalanced** (for example, if 99% of transactions are legitimate and only 1% are fraudulent), simple classification accuracy is completely useless. A broken classifier that blindly guesses "legitimate" every time will achieve a 99% accuracy rate while completely failing to detect fraud.

To diagnose the true performance of an algorithm, we analyze its **Confusion Matrix**. This cross-tabulates predictions against ground truth labels across four quadrants:
- **True Positives (TP):** Correctly predicted positive instances.
- **True Negatives (TN):** Correctly predicted negative instances.
- **False Positives (FP):** Negatives incorrectly classified as positives (Type I Error).
- **False Negatives (FN):** Positives incorrectly classified as negatives (Type II Error).

Using these metrics, we calculate **Precision** (the ratio of true positives to all predicted positives) and **Recall** (the ratio of true positives to all actual positive instances).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression

# Load the Breast Cancer dataset (binary classification target)
cancer = load_breast_cancer()
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(cancer.data, cancer.target, random_state=0)

# Train the structural estimator model
logreg_classifier = LogisticRegression(max_iter=10000).fit(X_train_c, y_train_c)
predictions = logreg_classifier.predict(X_test_c)

# Compute the categorical confusion matrix
matrix = confusion_matrix(y_test_c, predictions)

# Render the Confusion Matrix plot using Scikit-learn's modern display objects
display_engine = ConfusionMatrixDisplay(confusion_matrix=matrix, display_labels=cancer.target_names)
display_engine.plot(cmap='Blues')

plt.title("Confusion Matrix: Breast Cancer Malignant vs Benign Predictions", fontsize=12)
plt.grid(False) # Turn off background grid lines for clarity
plt.show()

## 5.5 Chapter Summary
Key takeaways from Chapter 5:
- **Cross-Validation** provides a robust, low-variance estimate of model generalization by evaluating performance across multiple structural data splits.
- **Grid Search** automates the hyperparameter tuning workflow. Remember to isolate your data beforehand via a secure train-test split to prevent data leakage.
- **Accuracy is a dangerous metric** when dealing with imbalanced targets. Always generate a **Confusion Matrix** and monitor precision and recall to verify the true predictive power of your algorithms.